# Downstream Retrieval And PCST

This notebook uses the main project config to load its downstream database.
In practice that means it reads the configured node table, edge table, and final GCN embeddings.

It is split into two modules:
1. Retrieve a full, unpruned subgraph.
2. Apply PCST to prune that subgraph.

In [186]:
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yaml
from pcst_fast import pcst_fast
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "configs").exists() and (REPO_ROOT / ".." / "configs").exists():
    REPO_ROOT = (REPO_ROOT / "..").resolve()

def resolve_path(path_str: str) -> Path:
    path = Path(path_str)
    if path.is_absolute():
        return path
    return REPO_ROOT / path

with open(resolve_path("configs/config.yaml"), "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

plotly_theme = config.get("visualization", {}).get("plotly", {})
THEME_AXIS_COLOR = plotly_theme.get("axis_color", "#475569")
THEME_TITLE_COLOR = plotly_theme.get("title_color", "#000000")
THEME_TEXT_COLOR = plotly_theme.get("text_color", "#334155")
THEME_GRID_COLOR = plotly_theme.get("grid_color", "#E2E8F0")
THEME_PAPER_BGCOLOR = plotly_theme.get("paper_bgcolor", "#FFFFFF")
THEME_PLOT_BGCOLOR = plotly_theme.get("plot_bgcolor", "#FFFFFF")
THEME_SEQUENCE = plotly_theme.get("sequence", ["#1F3A5F", "#2F5D8A", "#4A7FA7", "#6B9EC9"])
THEME_FONT_FAMILY = plotly_theme.get("font_family", "Arial, sans-serif")
THEME_BASE_FONT_SIZE = int(plotly_theme.get("base_font_size", 15))
THEME_TITLE_FONT_SIZE = int(plotly_theme.get("title_font_size", 30))
THEME_AXIS_TICK_FONT_SIZE = int(plotly_theme.get("axis_tick_font_size", 12))


# Load the downstream database from the main project config.
nodes = pd.read_csv(resolve_path(config["paths"]["nodes_csv"]))
edges = pd.read_csv(resolve_path(config["paths"]["edges_csv"]))
node_embedding_path = resolve_path(config["paths"]["node_features"])
# The example retrieval should query the raw text embedding space; node_features also contains
# metadata columns, so we keep only the encoder-width prefix for cosine retrieval.
node_embeddings = np.load(node_embedding_path)[:, :config["embedding"]["embedding_dim"]]

# Keep node ids as strings and align the embedding matrix with nodes.csv row order.
nodes = nodes.copy()
nodes["node_id"] = nodes["node_id"].astype(str)
edges = edges.copy()
edges["source"] = edges["source"].astype(str)
edges["target"] = edges["target"].astype(str)

# Build a direct lookup from node id to its embedding.
node_ids = nodes["node_id"].tolist()
node_emb_lookup = dict(zip(node_ids, node_embeddings))

embedding_model_name = config["embedding"]["model_name"]
# Query embeddings are generated from the same model family configured for node features.
node_texts = nodes["text"].fillna("").astype(str)
node_texts = node_texts.where(node_texts.ne(""), nodes["title"].fillna("").astype(str))
query_encoder = SentenceTransformer(embedding_model_name)
embedding_dim = int(node_embeddings.shape[1])


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Shared Helpers

In [187]:
NUM_RETRIEVED_SEEDS = 7
K_HOPS = 1
PRIZE_TOP_K = 10
EDGE_COST = 1.0


In [188]:
def build_graph(edges_df, node_ids=None):
    """Create a simple undirected NetworkX graph from edge rows.

    Parameters
    ----------
    edges_df:
        DataFrame with `source` and `target` columns.
    node_ids:
        Optional node ids to add even if they are isolated.
    """
    graph = nx.Graph()

    if node_ids is not None:
        for node_id in node_ids:
            graph.add_node(node_id)

    for row in edges_df.itertuples(index=False):
        if row.source != row.target:
            graph.add_edge(row.source, row.target)

    return graph

def get_text_rows(node_ids, nodes_df=nodes):
    """Return article / paragraph rows in the same order as the input ids."""
    requested = [str(node_id) for node_id in node_ids]
    order = {node_id: idx for idx, node_id in enumerate(requested)}

    matches = nodes_df[nodes_df["node_id"].isin(requested)].copy()
    matches["request_order"] = matches["node_id"].map(order)

    return matches.sort_values("request_order")[["node_id", "title", "type", "text"]].reset_index(drop=True)

def plot_subgraph(graph, title, nodes_df=nodes, score_lookup=None):
    """Visualise any retrieved graph with Plotly.

    If `score_lookup` is provided, similarity / prize values are shown in the hover text
    and the node size is increased by the prize value.
    """
    type_lookup = nodes_df[["node_id", "type"]].drop_duplicates("node_id").set_index("node_id")["type"].astype(str).to_dict()

    try:
        layout = nx.spring_layout(graph, seed=7)
    except TypeError:
        layout = nx.spring_layout(graph)

    edge_x = []
    edge_y = []
    for source, target in graph.edges():
        x0, y0 = layout[source]
        x1, y1 = layout[target]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(
        x=edge_x,
        y=edge_y,
        mode="lines",
        line={"width": 1, "color": THEME_GRID_COLOR},
        hoverinfo="none",
    )

    palette = {
        "article": THEME_SEQUENCE[0] if len(THEME_SEQUENCE) > 0 else THEME_AXIS_COLOR,
        "paragraph": THEME_SEQUENCE[1] if len(THEME_SEQUENCE) > 1 else THEME_AXIS_COLOR,
        "annex": THEME_SEQUENCE[2] if len(THEME_SEQUENCE) > 2 else THEME_AXIS_COLOR,
        "recital": THEME_SEQUENCE[3] if len(THEME_SEQUENCE) > 3 else THEME_AXIS_COLOR,
    }

    node_x = []
    node_y = []
    node_text = []
    node_label = []
    node_color = []
    node_size = []

    for node_id in graph.nodes():
        x, y = layout[node_id]
        node_type = type_lookup.get(node_id, "unknown")
        score = {} if score_lookup is None else score_lookup.get(node_id, {})
        similarity = score.get("similarity", np.nan)
        prize = score.get("prize", 0.0)

        node_x.append(x)
        node_y.append(y)
        node_label.append(node_id)
        node_color.append(palette.get(node_type.lower(), THEME_AXIS_COLOR))
        node_size.append(16 + (6 * prize))
        node_text.append(
            f"node_id={node_id}<br>type={node_type}<br>degree={graph.degree(node_id)}"
            f"<br>similarity={similarity:.4f}<br>prize={prize:.2f}"
        )

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode="markers+text",
        text=node_label,
        textposition="top center",
        hoverinfo="text",
        hovertext=node_text,
        marker={
            "size": node_size,
            "color": node_color,
            "line": {"width": 1, "color": THEME_PAPER_BGCOLOR},
            "opacity": 0.9,
        },
    )

    fig = go.Figure(data=[edge_trace, node_trace])
    fig.update_layout(
        title={"text": title, "font": {"color": THEME_TITLE_COLOR, "size": THEME_TITLE_FONT_SIZE, "family": THEME_FONT_FAMILY}},
        template="plotly_white",
        showlegend=False,
        font={"color": THEME_TEXT_COLOR, "size": THEME_BASE_FONT_SIZE, "family": THEME_FONT_FAMILY},
        paper_bgcolor=THEME_PAPER_BGCOLOR,
        plot_bgcolor=THEME_PLOT_BGCOLOR,
        xaxis={"showgrid": False, "zeroline": False, "showticklabels": False, "color": THEME_AXIS_COLOR, "tickfont": {"size": THEME_AXIS_TICK_FONT_SIZE, "family": THEME_FONT_FAMILY}},
        yaxis={"showgrid": False, "zeroline": False, "showticklabels": False, "color": THEME_AXIS_COLOR, "tickfont": {"size": THEME_AXIS_TICK_FONT_SIZE, "family": THEME_FONT_FAMILY}},
        margin={"l": 20, "r": 20, "t": 60, "b": 20},
        height=650,
    )

    return fig

# Build the full graph once. Both modules reuse it.
full_graph = build_graph(edges, node_ids=node_ids)


## Module 1: Retrieve An Unpruned Subgraph

In [189]:
def encode_query(query):
    """Encode the raw query with the embedding model from config.

    The returned query vector is used for both retrieval and PCST scoring.
    Its width must match the saved node embedding width.
    """
    query_emb = query_encoder.encode([query], normalize_embeddings=True, show_progress_bar=False)[0]
    if query_emb.shape[0] != embedding_dim:
        raise ValueError(
            f"Query embedding dimension {query_emb.shape[0]} does not match node embedding dimension {embedding_dim}."
        )
    return query_emb

def retrieve_seed_nodes(query_emb, node_embeds=node_embeddings, node_ids=node_ids, k=2):
    """Rank all nodes by cosine similarity to the query embedding and return the top-k node ids.

    Cosine similarity is computed in one embedding space:
    - query embedding: config['embedding']['model_name']
    - node embeddings: config['paths']['final_embeddings']
    """
    similarities = cosine_similarity([query_emb], node_embeds)[0]
    ranked_idx = np.argsort(similarities)[::-1][:min(k, len(similarities))]
    return [node_ids[idx] for idx in ranked_idx]

def expand_k_hops(graph, seeds, k=2):
    """Return the full k-hop subgraph around the seed nodes."""
    valid_seeds = [seed for seed in seeds if seed in graph]
    nodes_in_scope = set(valid_seeds)
    frontier = set(valid_seeds)

    for _ in range(k):
        next_frontier = set()
        for node_id in frontier:
            next_frontier.update(graph.neighbors(node_id))
        frontier = next_frontier
        nodes_in_scope.update(next_frontier)

    return graph.subgraph(nodes_in_scope).copy()

def retrieve_unpruned_subgraph(query, num_retrieved_seeds=NUM_RETRIEVED_SEEDS, k_hops=K_HOPS):
    """Module 1.

    Input:
    - raw text query
    - number of cosine-similarity seed nodes
    - number of hop expansions

    Output:
    - the full retrieved graph before pruning
    - the corresponding article / paragraph rows
    - a Plotly visualisation
    """
    query_emb = encode_query(query)
    seed_nodes = retrieve_seed_nodes(query_emb, k=num_retrieved_seeds)
    unpruned_graph = expand_k_hops(full_graph, seed_nodes, k=k_hops)

    retrieved_node_ids = list(unpruned_graph.nodes())
    retrieved_edges_df = pd.DataFrame(list(unpruned_graph.edges()), columns=["source", "target"])
    retrieved_text_df = get_text_rows(retrieved_node_ids)
    retrieved_figure = plot_subgraph(
        unpruned_graph,
        title=f"Unpruned retrieved subgraph for query: {query}",
    )

    return {
        "query": query,
        "query_emb": query_emb,
        "seed_nodes": seed_nodes,
        "subgraph": unpruned_graph,
        "subgraph_node_ids": retrieved_node_ids,
        "subgraph_edges_df": retrieved_edges_df,
        "text_df": retrieved_text_df,
        "figure": retrieved_figure,
    }

## Module 2: Apply PCST To The Retrieved Subgraph

In [190]:
def assign_rank_prizes(node_ids, query_emb, node_emb_lookup, top_k=10):
    """Assign node prizes from cosine similarity ranks.

    Only the top-k most similar nodes receive non-zero prizes.
    Higher similarity means a larger prize.
    """
    embs = np.vstack([node_emb_lookup[node_id] for node_id in node_ids])
    similarities = cosine_similarity([query_emb], embs)[0]

    prizes = np.zeros(len(node_ids), dtype=float)
    ranked_idx = np.argsort(similarities)[::-1][:min(top_k, len(similarities))]
    for rank, idx in enumerate(ranked_idx):
        prizes[idx] = top_k - rank

    return prizes, similarities

def graph_to_pcst_input(graph, prizes, edge_cost=1.0):
    """Convert a NetworkX graph into the array representation required by pcst_fast."""
    node_list = list(graph.nodes())
    node_to_idx = {node_id: idx for idx, node_id in enumerate(node_list)}

    edge_pairs = []
    edge_costs = []
    for source, target in graph.edges():
        edge_pairs.append((node_to_idx[source], node_to_idx[target]))
        edge_costs.append(edge_cost)

    return (
        node_list,
        np.asarray(edge_pairs, dtype=np.int32),
        np.asarray(prizes, dtype=np.float64),
        np.asarray(edge_costs, dtype=np.float64),
    )

def run_node_prize_pcst(graph, prizes, edge_cost=1.0, pruning="strong"):
    """Run PCST and map the selected vertex / edge indices back to node ids."""
    node_list, edge_array, prize_array, cost_array = graph_to_pcst_input(graph, prizes, edge_cost=edge_cost)
    selected_vertex_idx, selected_edge_idx = pcst_fast(edge_array, prize_array, cost_array, -1, 1, pruning, 0)

    selected_nodes = [node_list[idx] for idx in selected_vertex_idx]
    selected_node_set = set(selected_nodes)
    selected_edges = []
    edge_list = list(graph.edges())
    for edge_idx in selected_edge_idx:
        source, target = edge_list[edge_idx]
        if source in selected_node_set and target in selected_node_set:
            selected_edges.append((source, target))

    return selected_nodes, selected_edges

def prune_retrieved_subgraph(retrieval_result, prize_top_k=PRIZE_TOP_K, edge_cost=EDGE_COST):
    """Module 2.

    Input:
    - the output of Module 1
    - PCST prize top-k setting
    - constant edge cost

    Output:
    - the pruned graph
    - the corresponding article / paragraph rows
    - a Plotly visualisation
    """
    candidate_graph = retrieval_result["subgraph"]
    candidate_node_ids = retrieval_result["subgraph_node_ids"]

    candidate_node_emb_lookup = {
        node_id: node_emb_lookup[node_id]
        for node_id in candidate_node_ids
    }

    prizes, similarities = assign_rank_prizes(
        node_ids=candidate_node_ids,
        query_emb=retrieval_result["query_emb"],
        node_emb_lookup=candidate_node_emb_lookup,
        top_k=prize_top_k,
    )

    selected_nodes, selected_edges = run_node_prize_pcst(
        candidate_graph,
        prizes,
        edge_cost=edge_cost,
        pruning="strong",
    )

    node_scores_df = pd.DataFrame({
        "node_id": candidate_node_ids,
        "similarity": similarities,
        "prize": prizes,
    }).sort_values(["prize", "similarity"], ascending=False)

    selected_edges_df = pd.DataFrame(selected_edges, columns=["source", "target"])
    pruned_graph = build_graph(selected_edges_df, node_ids=selected_nodes)
    pruned_text_df = get_text_rows(selected_nodes)
    score_lookup = node_scores_df.set_index("node_id").to_dict("index")
    pruned_figure = plot_subgraph(
        pruned_graph,
        title=f"PCST-pruned subgraph for query: {retrieval_result['query']}",
        score_lookup=score_lookup,
    )

    return {
        "query": retrieval_result["query"],
        "selected_nodes": selected_nodes,
        "selected_edges_df": selected_edges_df,
        "node_scores_df": node_scores_df,
        "pruned_graph": pruned_graph,
        "text_df": pruned_text_df,
        "figure": pruned_figure,
    }

## Example

In [191]:
retrieval_result = retrieve_unpruned_subgraph(
    query="When is a general-purpose AI model one with systemic risk?",
    num_retrieved_seeds=NUM_RETRIEVED_SEEDS,
    k_hops=K_HOPS,
)

display(
    retrieval_result["text_df"].style
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "left")]},
        {"selector": "td", "props": [("text-align", "left"), ("white-space", "pre-wrap"), ("vertical-align", "top")]},
    ])
    .set_properties(subset=["text"], **{"min-width": "900px", "width": "900px"})
)


In [192]:
retrieval_result["figure"]

In [193]:
pruned_result = prune_retrieved_subgraph(
    retrieval_result,
    prize_top_k=PRIZE_TOP_K,
    edge_cost=EDGE_COST,
)

display(
    pruned_result["text_df"].style
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "left")]},
        {"selector": "td", "props": [("text-align", "left"), ("white-space", "pre-wrap"), ("vertical-align", "top")]},
    ])
    .set_properties(subset=["text"], **{"min-width": "900px", "width": "900px"})
)


,node_id,title,type,text
0,recital_111,Recital (111),recital,"(111) It is appropriate to establish a methodology for the classification of general-purpose AI models as general-purpose AI model with systemic risks. Since systemic risks result from particularly high capabilities, a general-purpose AI model should be considered to present systemic risks if it has high-impact capabilities, evaluated on the basis of appropriate technical tools and methodologies, or significant impact on the internal market due to its reach. High-impact capabilities in general-purpose AI models means capabilities that match or exceed the capabilities recorded in the most advanced general-purpose AI models. The full range of capabilities in a model could be better understood after its placing on the market or when deployers interact with the model. According to the state of the art at the time of entry into force of this Regulation, the cumulative amount of computation used for the training of the general-purpose AI model measured in floating point operations is one of the relevant approximations for model capabilities. The cumulative amount of computation used for training includes the computation used across the activities and methods that are intended to enhance the capabilities of the model prior to deployment, such as pre-training, synthetic data generation and fine-tuning. Therefore, an initial threshold of floating point operations should be set, which, if met by a general-purpose AI model, leads to a presumption that the model is a general-purpose AI model with systemic risks. This threshold should be adjusted over time to reflect technological and industrial changes, such as algorithmic improvements or increased hardware efficiency, and should be supplemented with benchmarks and indicators for model capability. To inform this, the AI Office should engage with the scientific community, industry, civil society and other experts. Thresholds, as well as tools and benchmarks for the assessment of high-impact capabilities, should be strong predictors of generality, its capabilities and associated systemic risk of general-purpose AI models, and could take into account the way the model will be placed on the market or the number of users it may affect. To complement this system, there should be a possibility for the Commission to take individual decisions designating a general-purpose AI model as a general-purpose AI model with systemic risk if it is found that such model has capabilities or an impact equivalent to those captured by the set threshold. That decision should be taken on the basis of an overall assessment of the criteria for the designation of a general-purpose AI model with systemic risk set out in an annex to this Regulation, such as quality or size of the training data set, number of business and end users, its input and output modalities, its level of autonomy and scalability, or the tools it has access to. Upon a reasoned request of a provider whose model has been designated as a general-purpose AI model with systemic risk, the Commission should take the request into account and may decide to reassess whether the general-purpose AI model can still be considered to present systemic risks."
1,article_52_para_3,"Article 52, Paragraph 3",paragraph,"3. Where the Commission concludes that the arguments submitted pursuant to paragraph 2 are not sufficiently substantiated and the relevant provider was not able to demonstrate that the general-purpose AI model does not present, due to its specific characteristics, systemic risks, it shall reject those arguments, and the general-purpose AI model shall be considered to be a general-purpose AI model with systemic risk."
2,recital_110,Recital (110),recital,"(110) General-purpose AI models could pose systemic risks which include, but are not limited to, any actual or reasonably foreseeable negative effects in relation to major accidents, disruptions of critical sectors and serious consequences to

In [194]:
pruned_result["figure"]

# Evaluation

In [195]:
import ast

from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.preprocessing import MultiLabelBinarizer

# Load the new gold file directly. Its columns already contain node ids in list format.
TEST_QUERIES_PATH = resolve_path("tests/test_queries.csv")
RAW_EMBED_PATH = resolve_path("models/embeddings/node_features.npy")
if not RAW_EMBED_PATH.exists():
    RAW_EMBED_PATH = resolve_path(config["paths"]["node_features"])

test_queries = pd.read_csv(TEST_QUERIES_PATH)

# `node_features.npy` stores [text embedding ; metadata]. We slice out only the
# encoder output here because queries live in pure text-embedding space and cannot
# be compared directly against the appended metadata dimensions.
raw_text_embeddings = np.load(RAW_EMBED_PATH)[:, :config["embedding"]["embedding_dim"]].astype(float)
gcn_eval_path = resolve_path(config["paths"]["gcn_embeddings"])
gnn_eval_path = resolve_path(config["paths"]["gnn_embeddings"])
embedding_variants = {
    "raw": raw_text_embeddings,
    "gcn": np.load(gcn_eval_path).astype(float),
    "gnn": np.load(gnn_eval_path).astype(float),
}
embedding_variants = {
    name: emb / np.clip(np.linalg.norm(emb, axis=1, keepdims=True), 1e-12, None)
    for name, emb in embedding_variants.items()
}

# Parse the stored list strings exactly as written in the gold file.
def parse_ids(value):
    text = "" if pd.isna(value) else str(value).strip()
    if text in {"", "[]"}:
        return []
    if text.startswith("[") and text.endswith("]"):
        text = text[1:-1]
    ids = [part.strip().strip("'\"") for part in text.split(",") if part.strip()]
    return sorted(set(ids))


def keep_ids(node_id_list, allowed_ids):
    return sorted({str(node_id) for node_id in node_id_list if str(node_id) in allowed_ids})


def align_query_embedding(query_emb, target_dim):
    q = np.asarray(query_emb, dtype=float).reshape(-1)
    d = int(target_dim)
    if q.shape[0] == d:
        out = q
    elif q.shape[0] > d:
        out = q[:d]
    else:
        out = np.pad(q, (0, d - q.shape[0]), mode="constant")
    denom = np.linalg.norm(out)
    return out / max(denom, 1e-12)


def rank_node_ids(query_emb, node_embeds, candidate_node_ids):
    if not candidate_node_ids:
        return []
    dim = node_embeds.shape[1]
    q = align_query_embedding(query_emb, dim)
    idx = [node_ids.index(node_id) for node_id in candidate_node_ids if node_id in node_ids]
    if not idx:
        return []
    cand_ids = [node_ids[i] for i in idx]
    cand_emb = node_embeds[idx]
    sims = cosine_similarity([q], cand_emb)[0]
    order = np.argsort(sims)[::-1]
    return [cand_ids[i] for i in order.tolist()]


def ndcg_at_k(ranked_ids, gold_set, k):
    top = ranked_ids[:k]
    rel = [1.0 if node_id in gold_set else 0.0 for node_id in top]
    dcg = sum(r / np.log2(i + 2) for i, r in enumerate(rel))
    ideal_len = min(len(gold_set), k)
    if ideal_len == 0:
        return 0.0
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_len))
    return float(dcg / idcg)


def ranking_metrics(ranked_ids, gold_ids, k_values=(1, 3, 5, 10)):
    gold_set = set(gold_ids)
    out = {}
    for k in k_values:
        top = ranked_ids[:k]
        hits = sum(1 for node_id in top if node_id in gold_set)
        out[f"precision@{k}"] = float(hits / k) if k > 0 else 0.0
        out[f"recall@{k}"] = float(hits / len(gold_set)) if gold_set else 0.0
        out[f"ndcg@{k}"] = ndcg_at_k(ranked_ids, gold_set, k)

    cutoff = max(k_values)
    mrr = 0.0
    for rank, node_id in enumerate(ranked_ids[:cutoff], start=1):
        if node_id in gold_set:
            mrr = 1.0 / rank
            break
    out[f"mrr@{cutoff}"] = mrr
    return out


# Reuse the notebook retrieval code, only swapping the embedding lookup for each model variant.
def retrieve_unpruned_subgraph_eval(query, node_embeds, num_retrieved_seeds=NUM_RETRIEVED_SEEDS, k_hops=K_HOPS):
    query_emb = encode_query(query)
    seed_nodes = retrieve_seed_nodes(query_emb, node_embeds=node_embeds, node_ids=node_ids, k=num_retrieved_seeds)
    unpruned_graph = expand_k_hops(full_graph, seed_nodes, k=k_hops)
    return {
        "query": query,
        "query_emb": query_emb,
        "seed_nodes": seed_nodes,
        "subgraph": unpruned_graph,
        "subgraph_node_ids": list(unpruned_graph.nodes()),
        "subgraph_edges_df": pd.DataFrame(list(unpruned_graph.edges()), columns=["source", "target"]),
        "text_df": get_text_rows(list(unpruned_graph.nodes())),
    }


def prune_retrieved_subgraph_eval(retrieval_result, node_embeds, prize_top_k=PRIZE_TOP_K, edge_cost=EDGE_COST):
    global node_emb_lookup
    original_lookup = node_emb_lookup
    try:
        node_emb_lookup = dict(zip(node_ids, node_embeds))
        return prune_retrieved_subgraph(retrieval_result, prize_top_k=prize_top_k, edge_cost=edge_cost)
    finally:
        node_emb_lookup = original_lookup


# Evaluate each embedding variant with both set-based and ranking metrics.
def evaluate_embedding(name, node_embeds, num_retrieved_seeds=NUM_RETRIEVED_SEEDS, k_hops=K_HOPS, prize_top_k=PRIZE_TOP_K, edge_cost=EDGE_COST, k_values=(1, 3, 5, 10)):
    rows = []
    for _, row in test_queries.iterrows():
        retrieval_result = retrieve_unpruned_subgraph_eval(row["Query"], node_embeds, num_retrieved_seeds=num_retrieved_seeds, k_hops=k_hops)
        pruned_result = prune_retrieved_subgraph_eval(retrieval_result, node_embeds=node_embeds, prize_top_k=prize_top_k, edge_cost=edge_cost)

        gold_recitals = parse_ids(row["Recitals"])
        gold_articles = parse_ids(row["Articles"])
        gold_chapters = parse_ids(row["Chapters"])
        gold_annexes = parse_ids(row["Annexes"])
        gold_items = sorted(set(gold_recitals) | set(gold_articles) | set(gold_chapters) | set(gold_annexes))

        pre_ranked = rank_node_ids(retrieval_result["query_emb"], node_embeds, retrieval_result["subgraph_node_ids"])
        post_ranked = rank_node_ids(retrieval_result["query_emb"], node_embeds, pruned_result["selected_nodes"])

        pre_m = ranking_metrics(pre_ranked, gold_items, k_values=k_values)
        post_m = ranking_metrics(post_ranked, gold_items, k_values=k_values)

        pre_prune_hits = keep_ids(retrieval_result["subgraph_node_ids"], set(gold_items))
        post_prune_hits = keep_ids(pruned_result["selected_nodes"], set(gold_items))

        rows.append({
            "query": row["Query"],
            "seed_nodes": retrieval_result["seed_nodes"],
            "gold_items": gold_items,
            "pre_prune": pre_prune_hits,
            "post_prune": post_prune_hits,
            "pre_ranked": pre_ranked,
            "post_ranked": post_ranked,
            **{f"pre_{k}": v for k, v in pre_m.items()},
            **{f"post_{k}": v for k, v in post_m.items()},
        })

    details = pd.DataFrame(rows)

    metric_names = [
        *[f"precision@{k}" for k in k_values],
        *[f"recall@{k}" for k in k_values],
        *[f"ndcg@{k}" for k in k_values],
        f"mrr@{max(k_values)}",
    ]

    classes = sorted(set(x for values in details["gold_items"] for x in values))
    if classes:
        mlb = MultiLabelBinarizer(classes=classes)
        y_true = mlb.fit_transform(details["gold_items"])
    else:
        mlb = None
        y_true = None

    summary_rows = []
    for stage in ["pre", "post"]:
        stage_name = f"{stage}_prune"
        entry = {"embedding": name, "target": "gold_items", "stage": stage_name}

        for metric_name in metric_names:
            col = f"{stage}_{metric_name}"
            entry[metric_name] = float(details[col].mean())

        if mlb is None:
            entry["f1"] = 0.0
            entry["precision"] = 0.0
            entry["recall"] = 0.0
        else:
            pred_col = "pre_prune" if stage == "pre" else "post_prune"
            y_pred = mlb.transform(details[pred_col])
            entry["f1"] = float(f1_score(y_true, y_pred, average="samples", zero_division=0))
            entry["precision"] = float(precision_score(y_true, y_pred, average="samples", zero_division=0))
            entry["recall"] = float(recall_score(y_true, y_pred, average="samples", zero_division=0))

        summary_rows.append(entry)

    summary = pd.DataFrame(summary_rows)
    return details, summary

In [196]:
retrieval_eval = {name: evaluate_embedding(name, emb) for name, emb in embedding_variants.items()}
retrieval_summary = pd.concat([summary for _, summary in retrieval_eval.values()], ignore_index=True)

# Keep only the core set-based metrics.
core_metrics_df = retrieval_summary[["embedding", "target", "stage", "precision", "recall", "f1"]].copy()

display(core_metrics_df.sort_values(["stage", "embedding"]).reset_index(drop=True))

# Percentage change vs raw baseline within each stage (robust version).
def pct_delta(value, base):
    if pd.isna(base) or np.isclose(float(base), 0.0):
        return np.nan
    return ((float(value) - float(base)) / abs(float(base))) * 100.0

pct_rows = []
for stage_name in sorted(core_metrics_df["stage"].dropna().unique()):
    stage_df = core_metrics_df[core_metrics_df["stage"] == stage_name].copy()
    raw = stage_df[stage_df["embedding"] == "raw"]

    raw_precision = float(raw["precision"].iloc[0]) if not raw.empty else np.nan
    raw_recall = float(raw["recall"].iloc[0]) if not raw.empty else np.nan
    raw_f1 = float(raw["f1"].iloc[0]) if not raw.empty else np.nan

    for _, row in stage_df.iterrows():
        pct_rows.append(
            {
                "embedding": row["embedding"],
                "stage": stage_name,
                "precision_pct_vs_raw": pct_delta(row["precision"], raw_precision),
                "recall_pct_vs_raw": pct_delta(row["recall"], raw_recall),
                "f1_pct_vs_raw": pct_delta(row["f1"], raw_f1),
            }
        )

pct_change_df = pd.DataFrame(pct_rows).sort_values(["stage", "embedding"]).reset_index(drop=True)
display(pct_change_df)


,embedding,target,stage,precision,recall,f1
0,gcn,gold_items,post_prune,0.0000,0.000000,0.000000
1,gnn,gold_items,post_prune,0.0000,0.000000,0.000000
2,raw,gold_items,post_prune,0.5000,0.207292,0.275000
3,gcn,gold_items,pre_prune,0.0000,0.000000,0.000000
4,gnn,gold_items,pre_prune,0.0625,0.062500,0.062500
5,raw,gold_items,pre_prune,0.6250,0.257986,0.344048


,embedding,stage,precision_pct_vs_raw,recall_pct_vs_raw,f1_pct_vs_raw
0,gcn,post_prune,-100.0,-100.00000,-100.00000
1,gnn,post_prune,-100.0,-100.00000,-100.00000
2,raw,post_prune,0.0,0.00000,0.00000
3,gcn,pre_prune,-100.0,-100.00000,-100.00000
4,gnn,pre_prune,-90.0,-75.77389,-81.83391
5,raw,pre_prune,0.0,0.00000,0.00000


In [ ]:
# Sample 5 random questions from the 16-question test bank
test_bank_path = resolve_path("tests/test_queries.csv")
test_bank_df = pd.read_csv(test_bank_path)

if "Query" not in test_bank_df.columns:
    raise KeyError("Expected a 'Query' column in tests/test_queries.csv")

sample_n = min(5, len(test_bank_df))
random_questions = test_bank_df["Query"].sample(n=sample_n, random_state=42).reset_index(drop=True)
display(random_questions.to_frame(name="Query"))


In [ ]:
# Run retrieval + pruning for the sampled questions and keep pruned subgraphs
if 'random_questions' not in globals():
    test_bank_df = pd.read_csv(resolve_path('tests/test_queries.csv'))
    random_questions = test_bank_df['Query'].sample(n=min(5, len(test_bank_df)), random_state=42).reset_index(drop=True)

sampled_pruned_outputs = []
for i, query in enumerate(random_questions.tolist(), start=1):
    retrieval_result = retrieve_unpruned_subgraph(
        query=query,
        num_retrieved_seeds=NUM_RETRIEVED_SEEDS,
        k_hops=K_HOPS,
    )
    pruned_result = prune_retrieved_subgraph(
        retrieval_result,
        prize_top_k=PRIZE_TOP_K,
        edge_cost=EDGE_COST,
    )

    sampled_pruned_outputs.append({
        'query': query,
        'retrieval_result': retrieval_result,
        'pruned_result': pruned_result,
    })

    print(f'\n[{i}] {query}')
    display(pruned_result['text_df'])
    display(pruned_result['figure'])


In [ ]:
# Generate LLM answers from pruned subgraph context using scripts/prompts.py
import importlib.util
import json as _json
from urllib import error as _url_error, request as _url_request

prompts_path = resolve_path('scripts/prompts.py')
if not prompts_path.exists():
    raise FileNotFoundError(f'Could not find prompts file at {prompts_path}')

spec = importlib.util.spec_from_file_location('project_prompts', prompts_path)
project_prompts = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(project_prompts)
build_prompt = project_prompts.build_prompt

def ollama_generate(prompt, model='qwen2.5:3b', timeout=300):
    payload = {
        'model': model,
        'prompt': prompt,
        'stream': False,
        'options': {'temperature': 0.1, 'top_p': 0.9, 'num_predict': 220},
    }
    req = _url_request.Request(
        'http://localhost:11434/api/generate',
        data=_json.dumps(payload).encode('utf-8'),
        headers={'Content-Type': 'application/json'},
        method='POST',
    )
    try:
        with _url_request.urlopen(req, timeout=timeout) as resp:
            body = resp.read().decode('utf-8')
    except TimeoutError as exc:
        raise RuntimeError('Ollama timed out. Try `ollama serve` and rerun.') from exc
    except _url_error.URLError as exc:
        raise RuntimeError('Could not connect to Ollama at http://localhost:11434. Start `ollama serve`.') from exc

    parsed = _json.loads(body)
    answer = str(parsed.get('response', '')).strip()
    if not answer:
        raise RuntimeError(f'Ollama response missing text: {parsed}')
    return answer

if 'sampled_pruned_outputs' not in globals() or not sampled_pruned_outputs:
    raise RuntimeError('Run the previous cell that builds sampled_pruned_outputs first.')

llm_results = []
for i, item in enumerate(sampled_pruned_outputs, start=1):
    query = item['query']
    pruned_text_df = item['pruned_result']['text_df']
    docs = pruned_text_df.to_dict(orient='records')

    prompt = build_prompt(query, docs)
    answer = ollama_generate(prompt, model='qwen2.5:3b')

    llm_results.append({'query': query, 'answer': answer})

    print(f'\n[{i}] Query: {query}')
    print('Answer:')
    print(answer)

display(pd.DataFrame(llm_results))


In [ ]:
# Simple faithfulness scoring: supported_claims / total_claims
import re
from sklearn.metrics.pairwise import cosine_similarity

def split_claims(text):
    parts = re.split(r'(?<=[.!?])\s+|\n+', str(text).strip())
    return [p.strip() for p in parts if p.strip()]

def simple_faithfulness(answer, evidence_texts, encoder, threshold=0.45):
    claims = split_claims(answer)
    evidence = [str(t).strip() for t in evidence_texts if str(t).strip()]
    if not claims:
        return 0.0, 0, 0
    if not evidence:
        return 0.0, 0, len(claims)

    ev_emb = encoder.encode(evidence, normalize_embeddings=True, show_progress_bar=False)
    supported = 0
    for claim in claims:
        c_emb = encoder.encode([claim], normalize_embeddings=True, show_progress_bar=False)[0]
        sims = cosine_similarity([c_emb], ev_emb)[0]
        if float(sims.max()) >= threshold:
            supported += 1

    total = len(claims)
    return supported / total, supported, total

if 'llm_results' not in globals() or 'sampled_pruned_outputs' not in globals():
    raise RuntimeError('Run the sampled-question and LLM-response cells first.')

faith_rows = []
for i, item in enumerate(sampled_pruned_outputs):
    query = item['query']
    answer = llm_results[i]['answer']
    evidence_texts = item['pruned_result']['text_df']['text'].fillna('').astype(str).tolist()

    score, supported, total = simple_faithfulness(answer, evidence_texts, query_encoder, threshold=0.45)
    faith_rows.append({
        'query': query,
        'faithfulness': round(score, 4),
        'supported_claims': supported,
        'total_claims': total,
    })

faithfulness_df = pd.DataFrame(faith_rows)
display(faithfulness_df)
print(f'Mean faithfulness: {faithfulness_df["faithfulness"].mean():.4f}')


In [ ]:
# Article coverage score: how many retrieved article nodes are in gold article labels
# - article_precision_in_graph = % of retrieved articles that are correct (in gold Articles)
# - article_recall_from_graph = % of gold Articles recovered by retrieval

if 'retrieval_eval' not in globals():
    raise RuntimeError('Run the evaluation cells first so retrieval_eval is available.')

node_type_lookup = (
    nodes[['node_id', 'type']]
    .drop_duplicates('node_id')
    .assign(node_id=lambda d: d['node_id'].astype(str), type=lambda d: d['type'].astype(str).str.lower())
    .set_index('node_id')['type']
    .to_dict()
)

def only_article_ids(node_id_list):
    return [str(nid) for nid in node_id_list if node_type_lookup.get(str(nid), '') == 'article']

coverage_rows = []
for emb_name, (details_df, _) in retrieval_eval.items():
    for _, r in details_df.iterrows():
        gold_articles = set(parse_ids(r.get('gold_items', [])))
        # Prefer query-level gold Articles from the source test table when available.
        matched = test_queries[test_queries['Query'] == r['query']]
        if not matched.empty:
            gold_articles = set(parse_ids(matched.iloc[0]['Articles']))

        pre_articles = set(only_article_ids(r['pre_ranked']))
        post_articles = set(only_article_ids(r['post_ranked']))

        pre_hits = pre_articles & gold_articles
        post_hits = post_articles & gold_articles

        coverage_rows.append({
            'embedding': emb_name,
            'query': r['query'],
            'stage': 'pre_prune',
            'retrieved_articles': len(pre_articles),
            'gold_articles': len(gold_articles),
            'article_hits': len(pre_hits),
            'article_precision_in_graph': (len(pre_hits) / len(pre_articles)) if pre_articles else 0.0,
            'article_recall_from_graph': (len(pre_hits) / len(gold_articles)) if gold_articles else 0.0,
        })
        coverage_rows.append({
            'embedding': emb_name,
            'query': r['query'],
            'stage': 'post_prune',
            'retrieved_articles': len(post_articles),
            'gold_articles': len(gold_articles),
            'article_hits': len(post_hits),
            'article_precision_in_graph': (len(post_hits) / len(post_articles)) if post_articles else 0.0,
            'article_recall_from_graph': (len(post_hits) / len(gold_articles)) if gold_articles else 0.0,
        })

article_coverage_detail = pd.DataFrame(coverage_rows)
article_coverage_summary = (
    article_coverage_detail
    .groupby(['embedding', 'stage'], as_index=False)[['article_precision_in_graph', 'article_recall_from_graph']]
    .mean()
    .sort_values(['stage', 'embedding'])
    .reset_index(drop=True)
)

article_coverage_summary['article_precision_in_graph_pct'] = (article_coverage_summary['article_precision_in_graph'] * 100).round(2)
article_coverage_summary['article_recall_from_graph_pct'] = (article_coverage_summary['article_recall_from_graph'] * 100).round(2)

display(article_coverage_summary)

